In [0]:
spark.sql("""
CREATE OR REPLACE VIEW power_prices.bronze.raw_file_index
COMMENT 'Every raw ENTSO-E response file in the bronze volume: dataset, series, period, size, fetch time, and whether silver has loaded it'
AS
WITH files AS (
  SELECT dataset, series_key, period_start, period_end, file_path,
         ROUND(bytes / 1e6, 2) AS size_mb, called_at AS fetched_at_utc, run_id
  FROM power_prices.ops.api_call_log
  WHERE outcome = 'ok' AND file_path IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY file_path ORDER BY called_at DESC) = 1
),
loaded AS (
  SELECT file_path, MAX(processed_at) AS loaded_to_silver_utc
  FROM power_prices.ops.silver_file_log
  GROUP BY file_path
)
SELECT f.*, l.loaded_to_silver_utc, l.file_path IS NOT NULL AS is_loaded_to_silver
FROM files f
LEFT JOIN loaded l ON f.file_path = l.file_path
""")

display(spark.sql("""
    SELECT dataset,
           COUNT(*)                                            AS raw_files,
           ROUND(SUM(size_mb) / 1024, 2)                       AS total_gb,
           SUM(CASE WHEN is_loaded_to_silver THEN 1 ELSE 0 END) AS loaded_to_silver
    FROM power_prices.bronze.raw_file_index
    GROUP BY dataset
    ORDER BY dataset
"""))